# MatPES MLIP Evaluation — r2SCAN SLURM Job Generator

Mirrors `matpes_run_generator.ipynb` exactly, but targets the **r2SCAN** functional dataset instead of PBE.

The generated scripts contain the same `get_static_energy` (ASE, no matcalc) and the same evaluation loop — no divergence between notebooks.

| Section | Purpose |
|---|---|
| 0 — Dataset Chunker | Split raw r2SCAN dataset into chunks (skip if chunks exist on cluster) |
| 1 — Imports & Helpers | Helper code embedded into every generated script |
| 2 — Potential Registry | Same registry with cluster env paths and `calc_setup_code` |
| 3 — Configuration | Active potential + chunking + SLURM settings (`FUNCTIONAL_PREFIX = "r2scan"`) |
| 4 — Generate Scripts | Creates run scripts + SLURM files + `mass_submit.sh` |
| 5 — Merge Results | Combines per-chunk output into `all_results_{mlip}_r2SCAN.json` |

**Two run modes** (set in Section 3):
- `"chunked"` — one script per chunk, submit all in parallel via SLURM
- `"single"` — one script loops over all chunks sequentially (one large SLURM job)

**Output files** after jobs finish:
- `data_{mlip}_r2SCAN_chunk_XX.json` — aggregated arrays
- `results_{mlip}_r2SCAN_chunk_XX.json` — per-structure details
- `all_results_{mlip}_r2SCAN.json` — merged (Section 5)

---

## Scientific Context

This generator targets the **MatPES-r2SCAN** dataset — the same structures as the PBE dataset but with forces computed at the r2SCAN meta-GGA level of theory. Evaluating MLIPs against r2SCAN forces allows direct comparison of PBE-trained vs. r2SCAN-trained potentials on higher-accuracy references.

Results from this generator are analyzed in `matpes_analysis_r2scan.ipynb`.

## HPC Requirements

Same virtual environments as the PBE generator. The dataset file needed is:

| Resource | Description |
|---|---|
| `MatPES-r2SCAN-2025.1.json.gz` | Raw MatPES r2SCAN dataset |
| Chunk directory | Chunks already exist at `/home/kamirian/scratch/MACE_env/jobs/matpes_chunk_all_data/datasetchunks/` |
| All MLIP venvs | Same as for the PBE generator (see `matpes_run_generator.ipynb`) |

## Workflow

1. **Section 0** — chunk `MatPES-r2SCAN-2025.1.json.gz` (skip if chunks exist)
2. **Section 1–2** — helpers and potential registry
3. **Section 3** — select potential (`FUNCTIONAL_PREFIX = "r2scan"` is fixed)
4. **Section 4** — generate and submit SLURM jobs
5. **Section 5** — merge into `all_results_{mlip}_r2SCAN.json`

---
## Section 0 — Dataset Chunker

Splits a raw MatPES r2SCAN dataset into N equally-sized chunks, writing the same directory layout used on the HPC cluster:

```
CHUNK_BASE_DIR/
  chunk00/  r2scan_00.json.gz
  chunk01/  r2scan_01.json.gz
  ...
  r2scan_indices.json.gz
```

**Chunks already exist on the HPC cluster** at:
`/home/kamirian/scratch/MACE_env/jobs/matpes_chunk_all_data/datasetchunks/`

**Skip to Section 1** unless you need to re-chunk with a different dataset or N.

| Parameter | Description |
|---|---|
| `CHUNK_DATASET_PATH` | Path to `MatPES-r2SCAN-2025.1.json.gz` |
| `CHUNK_BASE_DIR` | Output directory for the chunks (created if absent) |
| `CHUNK_N` | Number of chunks (default: 20) |
| `CHUNK_FUNCTIONAL` | Fixed to `'r2scan'` — do not change |
| `CHUNK_N_MAX` | Cap total entries for a quick test; `None` = use all |

In [ ]:
import gzip as _gzip
import os, json  # also imported in the main imports cell below

# ── Configuration ─────────────────────────────────────────────────────────────
# NOTE: r2SCAN chunks already exist on HPC — skip this cell unless re-chunking.
CHUNK_DATASET_PATH = "/path/to/MatPES-r2SCAN-2025.1.json.gz"   # update path before running
CHUNK_BASE_DIR     = "/home/kamirian/scratch/MACE_env/jobs/matpes_chunk_all_data/datasetchunks"
CHUNK_N            = 20
CHUNK_FUNCTIONAL   = "r2scan"   # fixed — do not change
CHUNK_N_MAX        = None

# ── Load entries ──────────────────────────────────────────────────────────────
print("Loading r2SCAN dataset (may take a minute)...")
_path_lower = CHUNK_DATASET_PATH.lower()
if _path_lower.endswith('.xyz'):
    from ase.io import read as _ase_read
    from pymatgen.io.ase import AseAtomsAdaptor as _ASEAdaptor
    _adaptor_s0 = _ASEAdaptor()
    _atoms_list = _ase_read(CHUNK_DATASET_PATH, index=':')
    all_entries = []
    for _atoms in _atoms_list:
        try:
            _forces = _atoms.get_forces().tolist()
            _energy = float(_atoms.get_potential_energy())
        except Exception:
            _forces = [[0.0, 0.0, 0.0]] * len(_atoms)
            _energy = 0.0
        _struct = _adaptor_s0.get_structure(_atoms)
        all_entries.append({
            'forces':         _forces,
            'energy':         _energy,
            'formula_pretty': _atoms.get_chemical_formula(),
            'structure':      _struct.as_dict(),
        })
elif _path_lower.endswith('.json.gz'):
    with _gzip.open(CHUNK_DATASET_PATH, 'rt') as _f:
        _raw = json.load(_f)
    all_entries = _raw.get('data', _raw) if isinstance(_raw, dict) else _raw
elif _path_lower.endswith('.json'):
    with open(CHUNK_DATASET_PATH, 'r') as _f:
        _raw = json.load(_f)
    all_entries = _raw.get('data', _raw) if isinstance(_raw, dict) else _raw
else:
    raise ValueError(f"Unsupported file format: {CHUNK_DATASET_PATH!r}  (must be .xyz, .json, or .json.gz)")

functional_lower = CHUNK_FUNCTIONAL.lower()
total_in_dataset = len(all_entries)

if CHUNK_N_MAX is not None:
    all_entries = all_entries[:CHUNK_N_MAX]
total_used = len(all_entries)
all_indices = list(range(total_used))

print(f"Label (file prefix) : '{functional_lower}'")
print(f"Total in dataset    : {total_in_dataset:,}")
print(f"Total to chunk      : {total_used:,}  \u2192  ~{total_used // CHUNK_N:,} per chunk")

# ── Write chunks ──────────────────────────────────────────────────────────────
os.makedirs(CHUNK_BASE_DIR, exist_ok=True)
base_size = total_used // CHUNK_N
remainder = total_used  % CHUNK_N
start = 0
for chunk_id in range(CHUNK_N):
    this_size     = base_size + (1 if chunk_id < remainder else 0)
    end           = start + this_size
    chunk_idxs    = all_indices[start:end]
    data_with_idx = [dict(e, original_index=i) for i, e in zip(chunk_idxs, all_entries[start:end])]
    payload = {
        'functional':       CHUNK_FUNCTIONAL,
        'total_in_dataset': total_in_dataset,
        'total_used':       total_used,
        'chunk_id':         chunk_id,
        'n_chunks':         CHUNK_N,
        'chunk_size':       this_size,
        'original_indices': chunk_idxs,
        'data':             data_with_idx,
    }
    chunk_dir = os.path.join(CHUNK_BASE_DIR, f'chunk{chunk_id:02d}')
    os.makedirs(chunk_dir, exist_ok=True)
    out_file  = os.path.join(chunk_dir, f'{functional_lower}_{chunk_id:02d}.json.gz')
    with _gzip.open(out_file, 'wt', encoding='utf-8') as _f:
        json.dump(payload, _f)
    print(f"  chunk{chunk_id:02d}  [{start:>7,} \u2013 {end-1:>7,}]  {this_size:>6,} entries")
    start = end

# ── Write indices file ─────────────────────────────────────────────────────────
idx_file = os.path.join(CHUNK_BASE_DIR, f'{functional_lower}_indices.json.gz')
with _gzip.open(idx_file, 'wt', encoding='utf-8') as _f:
    json.dump({'functional': CHUNK_FUNCTIONAL, 'indices': all_indices}, _f)

print(f"\nAll {CHUNK_N} chunks written to: {CHUNK_BASE_DIR}")
print(f"Indices file: {idx_file}")

In [ ]:
import os, stat, json, glob, textwrap

---
## Section 1 — Imports & Helpers

The helper functions here are identical to `matpes_run_notebook.ipynb` Section 1.  
They are stored as a string (`HELPERS_CODE`) so they can be embedded verbatim into every generated run script — ensuring the scripts and the interactive notebook share exactly the same code.

In [ ]:
HELPERS_CODE = textwrap.dedent("""
    import os, json, gzip, re
    import numpy as np
    from pymatgen.core import Structure
    from pymatgen.io.ase import AseAtomsAdaptor

    _adaptor = AseAtomsAdaptor()


    def load_json_gz(path):
        with gzip.open(path, 'rt') as f:
            return json.load(f)


    def save_json(obj, path):
        with open(path, 'w') as f:
            json.dump(obj, f, indent=2)


    def get_entries(raw):
        return raw.get('data', raw) if isinstance(raw, dict) else raw


    def safe_filename(s):
        import re as _re
        return _re.sub(r'[^\\w\\-_.]', '_', s)


    def angle_between(v1, v2):
        n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if n1 < 1e-8 or n2 < 1e-8:
            return float('nan')
        return np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)))


    def get_static_energy(structure, calc):
        from pymatgen.core import Structure as _Structure
        if isinstance(structure, _Structure):
            atoms = _adaptor.get_atoms(structure)
        else:
            atoms = structure          # already ASE Atoms
        atoms.calc = calc
        return {
            'energy': float(atoms.get_potential_energy()),
            'forces': atoms.get_forces().tolist(),
        }


    def _xyz_to_entries(xyz_path, n_max=None):
        from ase.io import read as _ase_read
        atoms_list = _ase_read(xyz_path, index=':')
        if n_max is not None:
            atoms_list = atoms_list[:n_max]
        entries = []
        for atoms in atoms_list:
            try:
                forces = atoms.get_forces().tolist()
                energy = float(atoms.get_potential_energy())
            except Exception:
                forces = [[0.0, 0.0, 0.0]] * len(atoms)
                energy = 0.0
            structure = _adaptor.get_structure(atoms)
            entries.append({
                'forces':         forces,
                'energy':         energy,
                'formula_pretty': atoms.get_chemical_formula(),
                'structure':      structure.as_dict(),
            })
        return entries


    def load_all_entries(data_source, n_chunks=20, n_max=None):
        if os.path.isfile(data_source):
            if data_source.lower().endswith('.xyz'):
                return _xyz_to_entries(data_source, n_max=n_max)
            else:
                loader = load_json_gz if data_source.endswith('.gz') else (
                    lambda p: json.load(open(p))
                )
                all_entries = get_entries(loader(data_source))
        else:
            all_entries = []
            for idx in range(n_chunks):
                path = os.path.join(data_source, f'chunk{idx:02d}', f'{FUNCTIONAL_PREFIX}_{idx:02d}.json.gz')
                all_entries.extend(get_entries(load_json_gz(path)))
                if n_max is not None and len(all_entries) >= n_max:
                    break
        if n_max is not None:
            all_entries = all_entries[:n_max]
        return all_entries
""").strip()

print("HELPERS_CODE defined.")

---
## Section 2 — Potential Registry

Mirrors `matpes_run_notebook.ipynb` Section 2, with cluster-specific additions:
- `mlip_name` — identifier used in output filenames
- `site_pkgs` — path to the venv's site-packages (injected into `sys.path`)
- `venv_activate` — path to the venv's `activate` script (sourced in SLURM header)
- `python` — full path to the venv Python binary
- `model_path` — path to the pre-trained model (`None` for built-in models like CHGNet)
- `calc_setup_code` — snippet that defines `calc` and is injected verbatim into the generated run script

**To add a new potential:** copy the commented template at the bottom, rename the key, and fill in your paths.

In [ ]:
POTENTIAL_REGISTRY = {

    # ── CHGNet ────────────────────────────────────────────────────────────────
    "chgnet": {
        "mlip_name":       "chgnet",
        "site_pkgs":       "/home/kamirian/scratch/chgnet_env/lib/python3.10/site-packages",
        "venv_activate":   "/home/kamirian/scratch/chgnet_env/bin/activate",
        "python":          "/home/kamirian/scratch/chgnet_env/bin/python",
        "model_path":      None,
        "calc_setup_code": textwrap.dedent("""
            from chgnet.model.dynamics import CHGNetCalculator
            calc = CHGNetCalculator()
        """).strip(),
    },

    # ── TensorNet (MatPES r2SCAN v2025.1) ─────────────────────────────────────
    "tensornet": {
        "mlip_name":       "tensornet",
        "site_pkgs":       "/home/kamirian/scratch/M3GNet_2/lib/python3.10/site-packages",
        "venv_activate":   "/home/kamirian/scratch/M3GNet_2/bin/activate",
        "python":          "/home/kamirian/scratch/M3GNet_2/bin/python",
        "model_path":      "/home/kamirian/scratch/M3GNet_2/pretrained_models/TensorNet-MatPES-r2SCAN-v2025.1-PES/",
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── MACE (MPA-0 medium) ────────────────────────────────────────────────────
    "mace": {
        "mlip_name":       "mace",
        "site_pkgs":       "/home/kamirian/scratch/MACE_env/lib/python3.10/site-packages",
        "venv_activate":   "/home/kamirian/scratch/MACE_env/bin/activate",
        "python":          "/home/kamirian/scratch/MACE_env/bin/python",
        "model_path":      "/home/kamirian/scratch/MACE_env/foundation_models/imporved_highpressure_accuracy_mace-mpa-0/mace-mpa-0-medium.model",
        "calc_setup_code": textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },

    # ── M3GNet (MatPES-PBE v2025.1) ───────────────────────────────────────────
    "m3gnet_matpes_pbe": {
        "mlip_name":       "m3gnet_matpes_pbe",
        "site_pkgs":       "/home/kamirian/scratch/M3GNet_2/lib/python3.10/site-packages",
        "venv_activate":   "/home/kamirian/scratch/M3GNet_2/bin/activate",
        "python":          "/home/kamirian/scratch/M3GNet_2/bin/python",
        "model_path":      "/home/kamirian/scratch/M3GNet_2/pretrained_models/M3GNet-MatPES-PBE-v2025.1-PES/",
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── M3GNet (MP-2021.2.8) ──────────────────────────────────────────────────
    "m3gnet_mp": {
        "mlip_name":       "m3gnet_mp",
        "site_pkgs":       "/home/kamirian/scratch/M3GNet_2/lib/python3.10/site-packages",
        "venv_activate":   "/home/kamirian/scratch/M3GNet_2/bin/activate",
        "python":          "/home/kamirian/scratch/M3GNet_2/bin/python",
        "model_path":      "/home/kamirian/scratch/M3GNet_2/pretrained_models/M3GNet-MP-2021.2.8-PES/",
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── UMA (FAIRChem / Meta) ─────────────────────────────────────────────────
    "uma": {
        "mlip_name":       "uma",
        "site_pkgs":       "/home/kamirian/scratch/Facebook/uma_env/lib/python3.10/site-packages",
        "venv_activate":   "/home/kamirian/scratch/Facebook/uma_env/bin/activate",
        "python":          "/home/kamirian/scratch/Facebook/uma_env/bin/python",
        "model_path":      "/home/kamirian/scratch/Facebook/Models/uma-s-1p1.pt",
        "calc_setup_code": textwrap.dedent("""
            from fairchem.core import FAIRChemCalculator
            from fairchem.core.units.mlip_unit import load_predict_unit
            calc = FAIRChemCalculator(
                load_predict_unit(MODEL_PATH, device="cpu"),
                task_name="omat",
            )
        """).strip(),
    },

    # ── Custom / New Potential ────────────────────────────────────────────────
    # "my_new_mlip": {
    #     "mlip_name":       "my_new_mlip",
    #     "site_pkgs":       "/path/to/venv/lib/pythonX.Y/site-packages",
    #     "venv_activate":   "/path/to/venv/bin/activate",
    #     "python":          "/path/to/venv/bin/python",
    #     "model_path":      "/path/to/model",   # or None if built-in
    #     "calc_setup_code": "from mypackage import MyCalc\ncalc = MyCalc(MODEL_PATH)",
    # },
}

print("Available potentials:", list(POTENTIAL_REGISTRY.keys()))

---
## Section 3 — Configuration

Same as the PBE run generator, with two r2SCAN-specific settings:
- `FUNCTIONAL_PREFIX = "r2scan"` — matches the chunk filenames `r2scan_XX.json.gz` on HPC
- `DATASET_LABEL = "r2SCAN"` — used in output file names: `data_{mlip}_r2SCAN_chunk_XX.json`
- `RUN_MODE` — `"chunked"` (one SLURM job per chunk) or `"single"` (one job for all chunks)
- `SLURM` — resource settings applied to every SLURM submit script

Each chunk file is expected at: `BASE_DATA_DIR/chunk{XX}/r2scan_{XX}.json.gz`

In [ ]:
ACTIVE_POTENTIAL  = "mace"       # key from POTENTIAL_REGISTRY
RUN_MODE          = "chunked"    # "chunked" or "single"
N_PARTS           = 20           # number of chunks
FUNCTIONAL_PREFIX = "r2scan"     # chunk filename prefix — matches r2scan_XX.json.gz on HPC
DATASET_LABEL     = "r2SCAN"     # label used in output filenames (data_*_r2SCAN_chunk_XX.json)
BASE_DATA_DIR     = "/home/kamirian/scratch/MACE_env/jobs/matpes_chunk_all_data/datasetchunks/"
OUTPUT_DIR        = os.getcwd()  # where code_chunk_XX/ dirs and scripts are created

# Set DATA_SOURCE to override the default chunked layout.
# Options:
#   None                    -> use BASE_DATA_DIR with chunked layout (default)
#   "/path/to/file.xyz"     -> single XYZ file (all structures in one job)
#   "/path/to/data.json.gz" -> single JSON.gz file
DATA_SOURCE = None

SLURM = dict(
    account       = "mogroup-paid",
    ntasks        = 1,
    cpus_per_task = 4,
    partition     = "standard",
    mem_per_cpu   = "4G",
    time          = "144:00:00",
)

cfg = POTENTIAL_REGISTRY[ACTIVE_POTENTIAL]
_data_source_display = DATA_SOURCE or BASE_DATA_DIR
print(f"Potential     : {cfg['mlip_name']}")
print(f"Mode          : {RUN_MODE}")
print(f"N_PARTS       : {N_PARTS}")
print(f"Data source   : {_data_source_display}")
print(f"Func prefix   : {FUNCTIONAL_PREFIX}")
print(f"Dataset label : {DATASET_LABEL}")
print(f"Model path    : {cfg['model_path']}")
print(f"Output dir    : {OUTPUT_DIR}")

---
## Section 4 — Generate Scripts

Builds each run script by assembling three parts in order:
1. **Header** — `sys.path` insertion, config constants (`MLIP_NAME`, `BASE`, `MODEL_PATH`, `FUNCTIONAL_PREFIX`, `DATASET_LABEL`, chunk index)
2. **Calculator setup** — the `calc_setup_code` snippet from the registry
3. **Helpers + evaluation loop** — `HELPERS_CODE` from Section 1 + the evaluation loop

In `"chunked"` mode: creates `code_chunk_XX/run_chunk_XX.py` + `code_chunk_XX/submit_chunk_XX.slurm` + `mass_submit.sh`  
In `"single"` mode: creates `run_all_{mlip}.py` + `submit_all_{mlip}.slurm`

**Output files per chunk:** `data_{mlip}_r2SCAN_chunk_XX.json` and `results_{mlip}_r2SCAN_chunk_XX.json`

In [ ]:
# ── Evaluation loop — r2SCAN version ──────────────────────────────────────────
# CHUNK_IDX is a fixed int (chunked mode) or a loop variable (single mode).
# DATASET_LABEL (e.g. "r2SCAN") is injected via the script header.
EVAL_LOOP = textwrap.dedent("""
    if os.path.isfile(DATA_SOURCE):
        # Direct file (.xyz / .json / .json.gz) — load all entries in one shot
        entries = load_all_entries(DATA_SOURCE)
    else:
        # Chunked directory layout
        chunk_path = os.path.join(BASE, f"chunk{CHUNK_IDX:02d}", f"{FUNCTIONAL_PREFIX}_{CHUNK_IDX:02d}.json.gz")
        entries = get_entries(load_json_gz(chunk_path))
    print(f" Loaded {len(entries)} structures")

    energies_mlip    = []
    forces_mlip      = []
    all_deltaF       = []
    all_deltaTheta   = []
    all_F_dft_mags   = []
    original_indices = []   # original dataset index for each structure
    results          = []

    for i, data in enumerate(entries):
        forces_dft        = data["forces"]
        energy_dft        = data["energy"]
        structure_formula = safe_filename(data["formula_pretty"])
        structure         = Structure.from_dict(data["structure"])
        orig_idx          = data.get("original_index", i)

        result                  = get_static_energy(structure, calc)
        mlip_energy_structure_i = result["energy"]
        mlip_forces_structure_i = result["forces"]

        energies_mlip.append(mlip_energy_structure_i)
        forces_mlip.append(mlip_forces_structure_i)
        original_indices.append(orig_idx)

        deltaF     = []
        deltaTheta = []
        F_dft_mags = []

        for f_dft, f_mlip in zip(forces_dft, mlip_forces_structure_i):
            mag_dft  = np.linalg.norm(f_dft)
            mag_mlip = np.linalg.norm(f_mlip)
            F_dft_mags.append(mag_dft)
            deltaF.append(mag_mlip - mag_dft)
            deltaTheta.append(angle_between(f_dft, f_mlip))

        for dF, dTheta, mag in zip(deltaF, deltaTheta, F_dft_mags):
            if not np.isnan(dTheta):
                all_deltaF.append(dF)
                all_deltaTheta.append(dTheta)
                all_F_dft_mags.append(mag)

        results.append({
            "original_index":           orig_idx,
            "formula":                  structure_formula,
            "deltaF":                   list(deltaF),
            "deltaTheta":               list(deltaTheta),
            "forces_dft":               np.array(forces_dft).tolist(),
            "mlip_forces_structure_i":  np.array(mlip_forces_structure_i).tolist(),
            "energy_dft":               float(energy_dft),
            "mlip_energy_structure_i":  float(mlip_energy_structure_i),
        })

    output = {
        "original_indices": original_indices,
        "energies_mlip":    [float(e) for e in energies_mlip],
        "forces_mlip":      [np.array(f).tolist() for f in forces_mlip],
        "all_deltaF":       list(all_deltaF),
        "all_deltaTheta":   list(all_deltaTheta),
        "all_F_dft_mags":   list(all_F_dft_mags),
    }

    tag = f"{MLIP_NAME}_{DATASET_LABEL}_chunk_{CHUNK_IDX:02d}"
    save_json(output,  f"data_{tag}.json")
    save_json(results, f"results_{tag}.json")
    print(f" Saved data_{tag}.json      ({len(all_deltaF):,} atoms)")
    print(f" Saved results_{tag}.json   ({len(results):,} structures)")
""").strip()


def make_run_script(mlip_name, site_pkgs, model_path, base, calc_setup,
                    data_source=None, chunk_idx=None, n_parts=None,
                    functional_prefix="r2scan", dataset_label="r2SCAN"):
    """
    Assemble a complete standalone Python run script from its parts.

    data_source    : str or None  -> if set, overrides chunked BASE path (e.g. .xyz file)
    chunk_idx      : int          -> chunked mode (processes one fixed chunk)
    n_parts        : int          -> single mode  (loops over range(n_parts))
    dataset_label  : str          -> label used in output filenames (e.g. 'r2SCAN')
    """
    resolved_data_source = data_source if data_source else base
    _ds_lower = resolved_data_source.lower()
    is_direct_file = _ds_lower.endswith('.xyz') or _ds_lower.endswith('.json') or _ds_lower.endswith('.json.gz')

    header = textwrap.dedent(f"""
        #!/usr/bin/env python3
        # Auto-generated by matpes_r2scan_run_generator.ipynb  |  Potential: {mlip_name}
        from __future__ import annotations
        import sys
        sys.path.insert(0, "{site_pkgs}")

        MLIP_NAME          = "{mlip_name}"
        BASE               = "{base}"
        DATA_SOURCE        = "{resolved_data_source}"
        MODEL_PATH         = "{model_path or ''}"
        FUNCTIONAL_PREFIX  = "{functional_prefix}"
        DATASET_LABEL      = "{dataset_label}"
    """).strip()

    if is_direct_file or chunk_idx is not None:
        effective_chunk_idx = chunk_idx if chunk_idx is not None else 0
        chunk_section = f"CHUNK_IDX = {effective_chunk_idx}"
        body          = EVAL_LOOP
        footer        = ""
    else:
        chunk_section = f"N_PARTS = {n_parts}"
        indented_loop = textwrap.indent(EVAL_LOOP, "    ")
        body          = f"for CHUNK_IDX in range(N_PARTS):\n    print(f\"\\n{'='*55}\\n Chunk {{CHUNK_IDX:02d}} / {n_parts-1:02d}\\n{'='*55}\")\n{indented_loop}"
        footer        = 'print("\\nAll chunks done.")'

    parts = [header, "", calc_setup, "", HELPERS_CODE, "", chunk_section, "", body]
    if footer:
        parts += ["", footer]

    return "\n".join(parts) + "\n"


# ── SLURM submit script template ──────────────────────────────────────────────
SLURM_TEMPLATE = textwrap.dedent("""
    #!/bin/bash
    #SBATCH --job-name={job_id}
    #SBATCH -A {account}
    #SBATCH --ntasks={ntasks}
    #SBATCH --cpus-per-task={cpus_per_task}
    #SBATCH -p {partition}
    #SBATCH --mem-per-cpu={mem_per_cpu}
    #SBATCH -t {time}
    #SBATCH --output=slurm_%j.out

    source {venv_activate}
    {python} {script_name}
""").lstrip()

# ── Mass-submit script (chunked mode only) ────────────────────────────────────
MASS_SUBMIT = textwrap.dedent("""
    #!/bin/bash
    set -euo pipefail
    for d in code_chunk_*; do
      [ -d "$d" ] || continue
      idx="${d##code_chunk_}"
      echo "Submitting $d/submit_chunk_${idx}.slurm"
      (cd "$d" && sbatch "submit_chunk_${idx}.slurm")
    done
""").lstrip()

print("Script builder and templates defined.")

In [ ]:
def _write_executable(path, content):
    with open(path, "w") as f:
        f.write(content)
    os.chmod(path, os.stat(path).st_mode | stat.S_IXUSR)


cfg            = POTENTIAL_REGISTRY[ACTIVE_POTENTIAL]
mlip_name      = cfg["mlip_name"]
model_path_str = cfg["model_path"] or ""
data_source    = DATA_SOURCE  # None -> fall back to BASE_DATA_DIR in make_run_script

# All output for this potential goes under OUTPUT_DIR/{mlip_name}/
potential_dir = os.path.join(OUTPUT_DIR, mlip_name)
os.makedirs(potential_dir, exist_ok=True)

if RUN_MODE == "chunked":
    for i in range(N_PARTS):
        code_dir = os.path.join(potential_dir, f"code_chunk_{i:02d}")
        os.makedirs(code_dir, exist_ok=True)

        run_path = os.path.join(code_dir, f"run_chunk_{i:02d}.py")
        _write_executable(run_path, make_run_script(
            mlip_name         = mlip_name,
            site_pkgs         = cfg["site_pkgs"],
            model_path        = model_path_str,
            base              = BASE_DATA_DIR,
            calc_setup        = cfg["calc_setup_code"],
            data_source       = data_source,
            chunk_idx         = i,
            functional_prefix = FUNCTIONAL_PREFIX,
            dataset_label     = DATASET_LABEL,
        ))

        slurm_path = os.path.join(code_dir, f"submit_chunk_{i:02d}.slurm")
        _write_executable(slurm_path, SLURM_TEMPLATE.format(
            job_id        = f"{mlip_name}_{i:02d}",
            venv_activate = cfg["venv_activate"],
            python        = cfg["python"],
            script_name   = os.path.basename(run_path),
            **SLURM,
        ))

    _write_executable(os.path.join(potential_dir, "mass_submit.sh"), MASS_SUBMIT)

    print(f"Generated {N_PARTS} chunk directories in: {potential_dir}/")
    print(f"  {mlip_name}/code_chunk_00/ ... {mlip_name}/code_chunk_{N_PARTS-1:02d}/")
    print(f"  {mlip_name}/mass_submit.sh")
    print(f"\nNext: rsync to cluster, then:  cd {mlip_name} && bash mass_submit.sh")

elif RUN_MODE == "single":
    script_name = f"run_all_{mlip_name}.py"
    run_path    = os.path.join(potential_dir, script_name)
    _write_executable(run_path, make_run_script(
        mlip_name         = mlip_name,
        site_pkgs         = cfg["site_pkgs"],
        model_path        = model_path_str,
        base              = BASE_DATA_DIR,
        calc_setup        = cfg["calc_setup_code"],
        data_source       = data_source,
        n_parts           = N_PARTS,
        functional_prefix = FUNCTIONAL_PREFIX,
        dataset_label     = DATASET_LABEL,
    ))

    slurm_path = os.path.join(potential_dir, f"submit_all_{mlip_name}.slurm")
    _write_executable(slurm_path, SLURM_TEMPLATE.format(
        job_id        = mlip_name,
        venv_activate = cfg["venv_activate"],
        python        = cfg["python"],
        script_name   = script_name,
        **SLURM,
    ))

    print(f"Generated: {run_path}")
    print(f"           {slurm_path}")
    print(f"\nNext: rsync to cluster, then:  sbatch {os.path.basename(slurm_path)}")

else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE!r}  (must be 'chunked' or 'single')")

---
## Section 5 — Merge Results

Combines all per-chunk `data_{mlip}_r2SCAN_chunk_*.json` files into one `all_results_{mlip}_r2SCAN.json`.  
Safe to run on partial results — picks up however many chunks are already saved.

In `"chunked"` mode the output files land inside `code_chunk_XX/` subdirectories, so this cell searches recursively.

In [ ]:
DATASET_LABEL = "r2SCAN"   # must match Section 3
mlip_name = POTENTIAL_REGISTRY[ACTIVE_POTENTIAL]["mlip_name"]

files = sorted(glob.glob(
    os.path.join(OUTPUT_DIR, "**", f"data_{mlip_name}_{DATASET_LABEL}_chunk_*.json"),
    recursive=True,
))

if not files:
    print("No chunk files found yet — run the SLURM jobs first.")
else:
    merged = {
        "original_indices": [],
        "energies_mlip":    [],
        "forces_mlip":      [],
        "all_deltaF":       [],
        "all_deltaTheta":   [],
        "all_F_dft_mags":   [],
    }

    for fp in files:
        with open(fp) as f:
            chunk = json.load(f)
        for key in merged:
            merged[key].extend(chunk[key])

    out_path = os.path.join(OUTPUT_DIR, f"all_results_{mlip_name}_{DATASET_LABEL}.json")
    with open(out_path, "w") as f:
        json.dump(merged, f, indent=2)

    print(f"Merged {len(files)} chunk(s)")
    print(f"  Total structures : {len(merged['energies_mlip']):,}")
    print(f"  Total atoms      : {len(merged['all_deltaF']):,}")
    print(f"\nSaved: {out_path}")